# 상세 이미지 OCR 실험 (Colab)

위에서부터 **칸을 하나씩 차례로 실행**하세요 (칸 왼쪽 ▶ 또는 Shift+Enter).

1. 설치 (몇 분 걸림)
2. 실험 파일 풀기
3. 3개 상품만 먼저 실행
4. 잘 되면 전체 실행
5. 결과 파일 내려받기

런타임 유형은 기본(CPU)이면 충분합니다.

In [ ]:
# 1. 설치
# paddlepaddle 3.3.x 는 CPU에서 오류가 나는 버그가 있어 3.2.2 로 고정한다
!pip install -q paddlepaddle==3.2.2 paddleocr pillow requests

In [ ]:
# 2. 실험 파일 풀기 (이 노트북 안에 들어 있는 파일: matching.py, data/, tools/ocr_experiment.py, samples/detail_images.json)
import base64, io, zipfile, os
BUNDLE = (
    "UEsDBBQAAAAIAMFISV2DpQ+/JxMAACg3AAALAAAAbWF0Y2hpbmcucHmtW31vG8eZ/5+fYo4BzrsJRUlO0+sRVXFuUwNBUdVogroF"
    "QRArcSVvRZEEd2nJVVXIDh2oiQ6RLauhXUpVWsWxXR+OiRS9HOQe4A9wH4S7+g73vMzMznIpWUYtIObu7MwzM8/r73lmks1mf5wX"
    "UedQhF/vRsf/JUbE6cfd/uGRCA8eRlv7/aNeTkTf7p92NkW0c/t082n4eRtaDh5E33YFdIi6O/D64kH44AF0WI9WO/lMJtq+G+58"
    "JURBXIm2Omp8/9tef+/EajTrM17VtWHYx7dP769aXuDO25nooCvH/KTfWxFh78/hpw9ExZsOhDXvBNM3ytyvf9iNjh9iP+gW7d6L"
    "jjsvD0/v7EZ/Xgs3ui8Pw+dH0fZXLw+j9m60sxltrYiKEzijb+d/69drIvpiPWrjwl8A9fCz3TytfaML7SJa/Xu0vR5+dSLCe+vR"
    "1ysi2vxU9spk5KrFhFjKNp3Aq/vZAjx+CP/m3/teTmSv49P4v8HTJD5d/vdleGw0vXmneQsaoCe8++50vVaRLZPYAu8zXsWtTbs0"
    "6j1o8vzyvLfoVqDhqlP13WUh3mJGTlcd3/dmblm2YHZlkCcC/nBZ+Fz2cNg4UKk580gym8/naR4ncGfrPHFQb/BiHGAJNly9ju+N"
    "eqNVdZpegJ3y34eWwJnFbRaBRmk5k81mMzPN+rwol2daQavplsvCm2/Um4FwarV6gFyp+ZmMbEN+c38UAC3d9dUA3ZQTM55brXDH"
    "hhPcqHpTqtM1eM1kPvrNtZ9+CBu0mIW01MmsnXn/ykdXyu9/8Ev4hB0tWBUIqFy28w2n6dYCMSqyOE02k8m8JUb+yT9tBKzaVkIe"
    "Uhp2JlNxZ0St3px3qt7v3LJUGqXyBVJnW4z8iB4KGZQccPWKJYmHB+3wq45NNkMUo602mM1K9HjrdLPT39sRoIzhxnOtu1v7Inz2"
    "JNxYxSH9ve/6va443XwCVojGg5/urWOv/jdHov9NLzrusjbjxLyTQkqf8/kcKjP+TOLPm9NjmnZE8HSwzqe4/nH8J9psh5+thk/3"
    "rbDXiTrPw8cn4JR2o94TEW58Y4vx8MsueKjt6KBDGzAJvTyUq3t5qGZE1wCm/glsObzTyYnwuB11n9BEW2vAHHAYm0/FGDTAlOJX"
    "TrXl/rTZrDcV4aizGu20kXHh47bVAE2ad7wa/iB1G1wg+idwEeHXayK8sx8erLBjQa5HH6/Kkfg12tgnUWmOvTyMWfXy0K+3mtOu"
    "wHH9vXZ0p0eOaKM34H6kotCvZN+EkFqVn3UDS8kwl2xtZG2bBkkWDY5Sch0YhrtVI5mhA+O0aAcHUqMc6aEpBMLzvZofOLBdixeZ"
    "k2ZQb4LfuGUF3KumNjYD7fROhm+zlfDGPd81pGVd0nalto/c+zAnrufEpIjaoPO9+8pGwBBON9vRowdgScDTS7zGm6CY6DyDgpy+"
    "GJQGVrCsNoOLNTZzMyem6vUq7WNgn/DJ8moB+LZq3Qls6nJT/FCMEembSBonzt/EvfiWfd4ms3qTSotXgA5G3o9vk6OgqBV98Tza"
    "PIEdooFneW8BuOQqbM5vzVvJ6dSOuMcPJ8TYay2go2yXLEjPBwSVmkmJEv8uJkCliWgJWoK79wRK604nsb1LZ6kXaZ+UysVm1XqM"
    "0wbNljs6g55q2HRNF0JeLeEtQWluglNgLpLS5AzhYjgGZifdp3zK6dWRZZvuNGFP8Qc76VDpd/nNRDaJqMDFAhDKZP5Dx2YLgvLv"
    "3NrER8AYO0NN4petquszcxdcb/ZG4HNUEwhSAC9gcGNYRAZAsIifJvmJzWnOvbVQb1b04LdUC44vAp2cQNAhOY84Zdg8wa0GqiUh"
    "PqZLvnWWu/IigWW1AhuioMEaNorTP/0x3HlORvRdN/rkL+AthPXjeuXWVS8AMLsJjDl9+IA88p3n0aOntqLoLF6EYri2ARRpzJRT"
    "RQ2VW5hgzGMBWHBa1aA840wHAM0m2C0CQYmnw95R+Jde9OmuEpEl6RCWBbRBcKMM/KlYCJ0KBIQG8MWCF9wQ9YZboy45AXGnXvFq"
    "sxPZVjAz8oOsLRzwuYa5sJ7jDHnYYsWaURPhW7mJ8rdQRcoVr8kzit8LP2jCthQkyyndKMfLkp1+LybrNYTR+ENLNTQKQpyiTICG"
    "d326thZtnSAUMmD76d01YMzpp0dx4DXnRLygxUJRuveg/7/A0tsn0SOKyCqcqgkVjFTv0uqdBfjALKbP5iQ2+iCzQbjgPGKKo4Sg"
    "y9yDZJa1iwyqS6YFUQByZtmjNOsLQ2IQv6NZwHcOlgvaw7Dmu1UKZcWb1Fd2kzNo1x8HIOigW0umhyN5WFofJIUJ+Rt7LmXCE8wd"
    "c9fqk96zasiWcoaikVkPGS6/6NHy3RzMdj5kLH+QQ+P+5AYm4D+L+JT84ixOwH+pL9LYhkximmGWtOCcz3l30fMD5D1px9Iyz6Cs"
    "CmTgwWBXWhb9W2AhkHlUYXARLKekTSRlFqjb/4Cs9f6uCD9fC3fgZ2+foOjOOmQPefKQOwCgnz0Nv9yiEdQt+tsJ5BQ5RMpR9wSh"
    "cHi8KsLHz9mytIlASJqqgqqhdrGqBDfQXmiteeZ4MevcrHuVMnxx/Rv1akWqOPBmWLf5esXNlsTEhMi6i9PVFrwBvqrIviQt8SOc"
    "JnZMahV5pwHeDPzS4IzWEjwu24iNADVEW0ciOj4i0A/7+9O+uPKrX3zwPvCkGz08ohLD5w+j7XvYlcE25AYSyUgBltFU0Tj1Gmb1"
    "E9kYLLaM8iF7opUr0TMmxyaAyEvLtja1FAE1lgglvoLR4EcL8/UygE9Ms6bqQVCfV28u+Ha3SW9JwrOKKi0ECOVActyFvYU77wUB"
    "4fqlWeVb/Hgb2oATfoOoYkemQWtcyJ01VHqnQnphOCBulloyq9MAVgHpGlEp9CdTLkkCwxUkRhmXluYWli+hPemocHoPq0UDIZaC"
    "/RefoPHIOtjS7LJkrl6/X28C76yElowoltrnaazGATFxqjdRrY0mVYUsmcGi3qIKy9Vtnae1OmScr4rx+jADOAOqv2L5UXcH6xWw"
    "WuBssHwJV/vh6PXRSZ3Md0ymASa/VUawFq9O+fiUlpypCq9anIaJpoTPk2eCZ3KBKbU8j0GvWAfMREjwwrySAViRfDPo/rJNOz++"
    "y+EGAQny3ML6YAEhGaTjrj8tHy8QfrhqGz67SzVW+JU1Va26qNFR7x+wNclzreJHALyEFR6shAdthu2QuKKCn4C0bB1vAneRQHJ2"
    "CRe5LJZwgctZgKMLblO6UHIB8eri0KS80txC+aKOCcQ8t6Cok4OCBSRFjaQgkAco4tnYE+hJDKXFhoQ8seHNyPJdO1WLlznAXK2+"
    "ULMGmHIhcaasZXs1thZAzjlMw8O978gbbe0D8h5iVRJsUGmt3+vCb1zOono7BV3Op7WYJXuKBj8xbcljKRiE5dN2bObtoDMoSfQE"
    "KbHED5X07hNl1yHMwEJBXIQldIDlhg2sobZP2z1ILT97AsoJDMca6t6OGI+6qNQr0bbS+mi7DTvFzDGGHcwT8uDbqxJZMQdMSCWP"
    "E86sFusqxzn46V9M/ARMlEOLcamglMrsqBb7mihODsUKWEIMxdlSUc+pqhylEpaUkLiWq6GfUg62wr9TkG+XZ7zg9cXHGTgnlFqM"
    "Y38Yj40jL9kuwrUVKpdRvRf7yxo4p3f/97eyDA70hNIV18qBeFtcL/KHEqZkoyqEgHrTaFUqmBAWEhoRP/9g0oZu1s+v/Fq9YToP"
    "D6PQREXewVIBaxEbCxE1lGfntlQeNJx76wKr+7KiycZGnvava9GjHurtf69A1puTqRFGHv9G06vNlecASataN9ry4xcievYCSWNR"
    "/bPj/hEmxIWBLcme78QVkRHZZgNjsBhu1eDrHG4RnibSYdd0JMgzO1HGvrAFkOpgEp7WIWUhphegXlp5VM+UFaASUJvfcGraCigZ"
    "xJOFOA2hTjW3FTSpmGqN5ceSHVDiRAQkTYcxOyrvAKZD4NtcIxRAAjMLl7TQ1LowRR3Pj+UEZqRj+CDntqkyZGiHTtpYKoY+Y8FX"
    "W6UsV5IGvy1S1itLDRzNiNMD9XfYXdWtMVuZ4f50venGSj+EFSy3Ad8iK5pSJwGLjmn5kYbGrEB5DMxpzquEAarJTSOaR0nNzJzL"
    "VxpLbugNRObv28kzcfZtEgq+vmtLh2kj8sSWRlnsJ/cxLWDFQE3jcihm+Tw9qjR4jv7xGmQMEL47J+AcOaSh9lw2CPNQMY6OBrYD"
    "5F8/YL3KXOkkbUKkowZ7ZGRbw5ueswKJSKse5FWyhuzPeQ2Dj8NYhX/TkDRSsWJ2ULd1NE1aAGgfJZpBOj9BpZ2lvKmUpJ/HLBBT"
    "64mqMz9VccRsQYwMIU5mi0kbOICtFXV+HK12CT10d0AomnC9hZ63mFjBoJHS5KVBt1HkczOCu0gFNuorQIpcKxULxEjJZZ6I+Izy"
    "yA2AgFhxsqW4ckDnO+fgCz6sQP/D+TH5j5wwWARyHBnmmOzieCnBhHfk4phkDi80QHPCoOH9zZjue7ast0soTdcoyDnQzRTj0H4o"
    "4JC1evP09kjebFEXYvhGy1Wv5lSp+Nbu9I96aHtZmgZcoT7/1nCIyt0biUo1qjSuSDpSGppgic9i91jyVGenkmM8SFHPxocHWHq6"
    "4QVceVQOCStOKacla08D7QNIGoE76vAcJ2KvUcDiQCANsJSIlEBzOIylw5Ive/1v9mWdjzCWrpUggnr0NK6V8OGBPBxtqB2hxgJz"
    "1J7tHL0au9XqrzeilkleA5cIH8S/0jhsLuqOpTTyiPE3UjT4kSIql2fSNbq/krTB3BRp3C1oMpOzE1MYw86dQjbg4aBSJqyglP0b"
    "DiQPZ9x5KZJPJ0OKY1uqCIe6L0W3I+sn/b0TOn/evZc3s7IVSskoTTUCG7S+PMTAhgD5mjy3BrRr4bWP8bjQIPXrIlnTUjpgFQQA"
    "ClkK9Zo+xqd/wgMWLsvSujy1T/eiSUqAbNKfeGJtNEPP9eU+iIqx9IH2M6eV59xqk8N6ylWorssqw5N1Uqfy25YfzLu12NYu4F9e"
    "DZaCVgOWQTqV43HYsZSoWAEwwks/Hbq8wFAJXPV5itfFQxQ8BO7dB93LyfZwvavPy+k05fUBUsWtBk5OOI1G1aMKPKFRo3qFioQ2"
    "RInzEJOyh5fWzcOEcw4isCGgoJ6qdQ4EgzgMJGRjp4ugtCUM2rxuTjOKWTo8yJZS3eXWVcl0KYubzBZw4+raIhOgF9AASamQpIv3"
    "0HA+bK+3gBC95cS79nIiLCYZLpUS4APEQRnnYZ9Ku3ij6u1CJQitZ5a6kkNhHwxVKhy/Y3J0GRr7e0/63z4N99qG+rGqCQvP7dqr"
    "2PLOWH58DEb0e+swXIzA63sQmcb+QFfh6J7qUXyXDzYwqoPZUuLeZk6hjAIjEBF21jFKxliggFctloV5zVbe1UX3idW9tRWa9D+f"
    "0M23daziMYfTBrQirr1/VfwgevYCC4V0LDqauLWQH3DU4KXZVxs2yA7bumaDYz+9fSITHmmCydtwPkjWn8J8w8BtwA7gldkiA51y"
    "kL6DKIkQEqLmKfV23jUolqJMvfmiH11fTpSYbIIdBAk1FgRVwBH6npTUxzIserhrZKMlIZKhKq1Mtw9UOc5PeC3Y9TuwWaWItA5d"
    "h2uC856HFVTKbBo+rkQ5YnKoahkDjefYSC5l+vwHM7jNBJnkvRD2PDXK+qDp3TjRo84Jxw6K8zPKV9n2FoXZyAZIZbIve6h0m/vq"
    "cg3NAH3ocIKPw3dX6HxCKbAtjWtE/AwWIYsY0m/g6Lm8GNQJqQxgIrZUDzqy+58V49Zqv8fYlNZLeoRXhcASTh+t0U2Vu0fRQVfW"
    "yuWYCG8u9k7bvbjQjpOiXUC+LksA43I3/d5tQwWNg5fENX34RneE8b7t7UGIbCeNbLCUU8yabDBT9ynXDyhtSkh3qPzwr+U7UxQz"
    "ix7FMA8DGA2neGSmYraZ16TyXwm4mF4CcXmAuJKEAGvNlbTfLE/r6DaNFokbQM1Hq8NnqfPyGq0DYCh51sRnPkgnGY6n6FRREk6G"
    "zBQCGOYGhjiBi9m/+lPlsiGOIOErbXAFCVdpDzoIkypxQEVurr/lMHBP6f3YBqtSRZJFkMZicYyrIqYiP1TFzGH1EVWzcJtKQuxB"
    "0HkXETuPq2MKv1WFxbR8t4K6yaLifC4+jh62aBQWrTh5Ol7U/1NFCXtoukl5glEEXq3lGmot++WdSsUyqdiG4uJaDQxUbyAEMvrq"
    "bBnap8z2M7yqyBJXoDf9ApcTy5+RHKSSAHnZM+kowCBhFfPre3gFVWoqfJOMW7ZNhmHVljdm490e9uKJaaaarjNnxinu/mZKOT8h"
    "h8flFTx66GwC1ufwFmMbo6rzaoAX36OUd4HiWgrZYXyVmo6BB1GEHKQP3fGfGNbGtPCAG+sheM3XMGVlwokzsnOjfnxVyfjfgXjW"
    "IfqjgCKRjVsNZMiyjz+pYxb1SW3Z6KJu6wH+JK1L9GPtM+viqe0wKZU8xnLz03HlgoV0I+iocmlCHVJL4FgU6GBUyvw/UEsDBBQA"
    "AAAIAKkhSV0Sxazh+QIAAK4GAAAUAAAAZGF0YS90YWdfd2VpZ2h0Lmpzb26VlFFP2mAUhu/9FSdcaaYoJMsS7pYsy3ZjluzCS2Ok"
    "aCe2hta5xLCw2RkCOHTALK51GBE06baCZNSN7cKf0+/0P+wUFKgrul31tOn3nPe855xvcwwgMC+IMheIQMBRFTw0AN8aeHAGWN1D"
    "3QLbTLnRhQqh+7apBQGbZZZXYCr0+l44CHa7YzfMCDi5FGDZAqyUWM0AZlioaKir9IOGF2VnT2MnHXC2NLtt4f4uoPrVtlpE/4B6"
    "i52Z7mH2PQV2M4U1HfUUOCUV6wQtZbCksmw1OEC5gF5iUge4v80yxSv2OFNOmN4h5ZdtZz9NWlCpXLZZQWNHXzCtsJ3iZRs/5TDb"
    "AFZoQChERU2455lyfCMfM4v2b4tVquysBSSTisyh3mH1HNjnFbzQun89e/QY8FDBrEbFQPgB8VyczEmyNB3jX8nrCU6aXovG5jc4"
    "fmlZloIvJFGAbg1EIb/yHQKkXVhg0m2IvLAkUT82Kaa3GC9HpDi/Sl8ANgPP3Wd4EgJzbjBDwSwFUyEKpMSi28ae9+RfHjBjsFon"
    "kJwcoMSXXGII1T04dx3MXsN9UGqVNZVpdnyKtXSfGOeEyGJCXBsWN3MtLjxanG1ZZKht7oLzxqDOeIBxUVjyk9iv1kfiAMiOGn2a"
    "xMcjG3yU85N3W8FYTrljWdCcnIFWZRxLCr0777QJD/phl3tL5d0vN7ws5lgljR9bmKniz6qTsUbRnz65wo9qug+doER0ExAwY42W"
    "H+XkSGI9Fou73tz0+ZbG4eEJZnVg9feo58bZZ4PqASdvOKXT3tK2UNEHaVYX5MiiKMui0E8Df7fTJ812zr2KaPE8qCgn9BZhpCU+"
    "vbxC7Zgsq3poZPLK8GT8LczH4B4Ns1UaXLpJfuzS4rNvFjXUW/QyH4u5Vd8cjdA/sDUnb9Kd1TxlzXO6xnY6HnScFzhhyISZ/zCh"
    "nibNv4qskGJHdD23KIvhTqKzVfLk2BDFuO9O390zPLA8qBWBl323OXR3+7NpGuJAkmDJseTYH1BLAwQUAAAACADnIkldmy2kNXED"
    "AABqCgAAEgAAAGRhdGEva2V5d29yZHMuanNvbo2WQXPaVhDH7/4Ub7jYnolpcuiFQ2dyaz9DJ+NR4WEUy5JHUoxJJjOyTToEyITE"
    "kJAY3HiqYJxRJgTThLS0h34cdvUduk+AoM57OBfgr99fy7J6u8uDFcYSm6bl8kSKJfDoIHxegneP/v2MRZ/e8WUNiy2GvX+wXWTh"
    "oY8nVThusY0fWHjUGn8esjX8tQrBEE9rDD558KnIJgpfDhi2R/iluZ5k456H/jP6HD5rrbna1mae61s5N3nXscx1+g6GpyUo19mt"
    "78e91jQynFcZVD1KCdsDqPhJtoqvh6vY9qIPYXkI7dEqg+OP48sRG/e/4CXFaZzCc5+N/wgoE8oC3o4orVfY9KIIeNQM61U4a5Hf"
    "w047bDThoseg3IROjxxsbbfg5iyTuZZlON+lrT1ua1s8uVtYTyZuiFJt80LesjMOVesBabqC5QA6I9I/J7K6m3IMfSdx58aERUJK"
    "6ILLM3KGTR/6xbDxNMYijysYDwOqC3ZKchP83lWzzgDePMayT09S4eg1sKJgQjj6fa6mdFEODctyFPfZ3ND2FwryP/iLtV+Qk/Ag"
    "gLP3ETO4mUrb1m7MIiElcP529vMEMyxzK2Y72r4uJ3D2kWqGf/l09JQOdr0FT2phfUS+JUGuteDf9eVpLOH4yhMtNX34jm6k8nqG"
    "xzgSUqJRx2F7GMPb85A0JyikCmsbhm5yCRDd+KaELwYSljU0W3aPmCqntRj89OOclEXhRcD2YPbrr1gc19bE6JEx+C2gfCKS4W7K"
    "vpfNGnwh1SI1jopOpYJmbd0wVDB6VJMBsqO5qbTlupY5p6UWXAxUlBC7dfOmCk+l6uYnNPOaMc1wc3EI9d9Dr47nnoJPlJzd5Zrp"
    "qOJWfGraGNIj2J6zP2vUl/BhODsQXzkiISVpzcnt8OmB+TpypRU+7c0LkdOz2cUiPw6WYeh3oX+pxDOt4nvc2OOuOjqtnmI3xqJT"
    "FmGghnRy4NijdUbnXWEZX3bhiQpCMBAtc9RQ8ImSszRtwXu2VVCl9nrIYpSnbbpI7ijJupJsyglc1EWJHg3JIndEQn7vuy6eHyhY"
    "pTSbIIJtm7TTZiwSUkL/J8RSPumK13K8BfKa7ripHM0difMbbNFcjXe9xCYEi64vc2xc62C2Pl3QEoPI9NCnwsSz09nWTbMwb86J"
    "vEoJPlx5uPIfUEsDBBQAAAAIAK0hSV387wN5IQMAAOkGAAARAAAAZGF0YS9yZWFzb25zLmpzb259VMFu2lAQvOcrnjin+QAOlXJr"
    "L7300GMVJaZFIXYFtKkURYLgRC6QhqgYEMKEKCQmEQcXaOpI9Ie863/o7rPBNiWVfLCf/GZnZ2b3aEOI1HtVKyqptEj5lZ732xVf"
    "BfYGfscUL14KfGzipCfQmtGZgLGL/dst4TklHF7iU4deBLYu0eig5Qp8csGuC9/Usdv0zQ7UhlsCq0O/UkJrDnZZ+Ge/4LSB7YaA"
    "pgO3bohI8HDvCL878qZzuJ0Lb1LCO8s3e/RWQYuKmDrUDAZMbTLnvLJT0NQCsT6iTzrIZIvpQi57QCdCHKXecj9YHcPdnFF884Jq"
    "CG9qMxg8UGFrTpTR0oX31IQy0/UmdYEnMxgMcdihDlLHmxG29kXJh9hvJHZ7zIpg34Bqc4FPwGh1BHATI8bGPjXrkARw9YfhPaeB"
    "NXcFPqeo6d289imEfyfhbcnUc51AHuG3Dbgboz6QsJ0xsRQkB52xB57rYovqT2fMawU7p6kf4tQ9dxYDRt2F615MD7IQfgwZDk0D"
    "ByZxXrSJVPBmhO0b2ezFPF6pkM2lD7N7SrwS6N/oYQMSN3Eyo3hxLf+7S0VC7deKw7DbEjMSZ3tBdiELR4adIzGmlNLHEvzowRUJ"
    "wZZfL0SjDLdma+Bfvwrxw9hQSpgOtEb8v13izqOKocdol0PcRH5CB5JF9pRiOv85k8mxOGEPRM9v1oNQSoaDMo9QWIcHJN5FlCg4"
    "p1KrXRzsFNO7WrGoqbKA7ALuSeizOvZlQKA2A+cnkySBFox5rl2slKU6AeKmCJ1L3I5smg7imQ64VRtwUo9hxFntKWowkwtW5w7U"
    "OiygX9H5YRK2wdbLLA7g4ZTPojhi/5qGf5Xe/2FoN9XG8nI8wXK+42hxohSE/fjuAHsIZUPiLmWIdFszLess+ZjNZNiT0HPCgMmc"
    "NYsZk7D5UceeTrPJgX7W6VxWVdRA02AVrRgSCZHYR3RG25UUoPAS3a7Li24F+lDTcokBvtD9M0eq0HXjCv+j63OI+2q2GF9syX5r"
    "hl91pbTLMX1+JSzNX96KlrCcvYdgnyfWCDE53jje+AtQSwMEFAAAAAgAriFJXaicKMpUAQAA2QEAABAAAABkYXRhL2NvbmZpZy5q"
    "c29uTZDLasJQEIb3PsWQVYKXqt0JPksITYqi5ojGYukFg1KkFqzFtJEmomBr25W0gUaIL5QzeYee3Nruzvnnm39m/osMACeqRFO4"
    "CnDSGanLYovISgWU/kmzJys8nbyh7QWmgbZD3x0Inkf0ywFcWbhwcxC3aLWO0q2RpgyMCgwTcO+yF+ByjI8OGh7Q6QKXMwEuoa2o"
    "UlM752MHwKebiFzd49ik2zugOxNNXSiAvxvgZoZ75va5oNMR+0O+dJ0tU9sD3A58d4z2IPUrQLfWqasNsVGBYGj5327I40oPGRzq"
    "wQOjzXTOC3PYzOjrAfDjgJZH1xbQyd53d/R2ziYbbB5fTE6m+pytE3dCNW3MAp9I+UQSoA/qEa9mG0IO1Gq8BTCiwOXCnP/iDcNO"
    "9v5f+o2R1fPHUSE9iimlSOgoUpeoYkvqM6kcSaSnndY1USPtiCsXM1eZH1BLAwQUAAAACADmIkld4zNHP2ACAAACBQAAEQAAAGRh"
    "dGEvYmFsYW5jZS5qc29upVLfS9tQFH73rzjkSVmtOhiMPI+hL2Owhz0MkW7GNbNtpIkWKYVWwyi1YIqNplviWiizlQ7iDGsL9cU/"
    "J/fc/2EnMf7aapns5XLvPed+33e+7+anAISVjKJJgggCXtRZvQ7MHbBvLlba4PdtHDZg+vWLl/AcTy9m4oB2k1smO9dZy76us9Mu"
    "aznoeIDHZf+8GYOtRGpT8t0i4GENyxbresDOPaaXA1R0qtGd7xrYNOgh22sTdMv1zzwRNGVjJZHZnsbdEjoW8F3b7w8A2zXgpsV2"
    "rJkYvFc0TUmHXXT3UJcka0kpe4112afC3QaYBWyW+HE1YkZHp1E6rGYAO6li6yzoI5XUhF+6pO+o5w9c1G3AA5dV6hFnsKNRCRiH"
    "Fq/ZeGRQ62c89Ng+ARoWnhQJD9al7ZySXVXjn1QlQx1GgMR+NQJa8iPwjt6gOSJ9dmgIt0bc7JLd6BRhAW39sv+U1iiDyP1Q9VBH"
    "u0MkvDSKtHDTC5WG2iNAIRbEnd1MSSrFnacDHd/Q9l24BcgLmUT66ivc9WtpbhGYM0JnAHy/x82OEAPh1twAQFDllLi0KCxTJQyf"
    "LufjC/OF2N/QVpv91Lm5D1HATwAbxeATHNhwlWaAH/2CEHxN1kRlS8qG8LfZ3xDn5FXpHvUscT8rhNTLVwqEt2Pn5KUea/24VRLy"
    "e/i1E6yVNq8M/tSSkjLih6yyMU5LLiGrmpiUPyb/yYhrj6NPd2PMY8a/V5sw/qvxMU/IYg4p8mPjfyKh0iP+RaXHvo/uCCH7d9ps"
    "rzxBgZqS0w8pUNflDB0f8oXWwlRh6jdQSwMEFAAAAAgAAmhJXRbtySbaCQAAwRQAABcAAAB0b29scy9vY3JfZXhwZXJpbWVudC5w"
    "eZVYbW/b1hX+rl9xwX4w2di0knTFJlQB0tRbgnWJ4SX74gkELV5ZrCmSJanKqmfASbTAq53FbeLGS+zARZ226TLMddTOwZw/JFL/"
    "Yc+5l6Qkv7SYgUTivfe8Pec55x5KUZTk7u2kc8iSnW7878Pk2xWW3LvPeocrybMN9caVGa23v8L6d7d7r/+aPN5g8YNO8nQFh1m8"
    "sRV/9ogE4lddfGPJ2l7/8ZbO4jtdNn0Fh5POtlj88m/9ze14bU8vFJK9lfh1p8QY822f2W4YmY7DfNOyHO5Vg/Sb/B9HHMdrsYB/"
    "3ORhFBakLiHcjuqeyyLPc8JJyBl80eeB3eBupPttduJPDc2G7/Bw0uKRaTuG3TDneah/FEJJcudl8uSFVsjO/oLuiQnHbtgRu4Bv"
    "C5z7pP1Cb387/nZ9nCXPtuKdo/gO4Pxqv7d/myW7wPG5Vij0N7cERjtHpQHU8fMjQive3Yv3nyabHTZxCch3GfIRf7U9dI4khcJk"
    "+0jsPNuIn3d7r3ZJAuD2fvxX8mWXIV20kOy86T9bFznc6WD7RfJqIw+PjZHabzrwd4zF6yvZ8zkIQ2KM1Pfv7CVP1+OH2yLx/zkU"
    "fr7u9H7cSBM5YmKFZegSWgEPm06UQgvCSAT0wkjQGT3AJhZ//98hEBDfiy4gunu7/8Uq6/99C3IQyth0jKukK/n2drL7eWqnv9nN"
    "Ix32SiZ88j2p17YuTbL+g258/5HgdOdrIIy4mCoXifLJs1XpDEu+PkrurRNWydouAxysv76ORBK4ycMu5DSEd/DPlOsESPz9d/H9"
    "QzqQRxZ/tgFqMPr45qXAsHew33t1RLmN97cAdrK2fSx18eqqTDjh1P/HZrJz2N+ECwd31WR3Ewu0nqxuJa8luR7fk84oilKoBV6D"
    "GUatGTUDbhjMbvheEDHTdb3IjGzPDQuFbC2Y980g5Nmz7WXfKIvZ97AdSqW+GdUdey7TOI3HQmHmxo2brCweVFi1HdjUdJDBcz7h"
    "qqbDgCgg8VGALp3U6OgAPIjU4jgLo0AlJZqW+9Uwo2rdducZe4u53sdmiU29U7xQKFy5euv6742rMHf+3WJRJvutrOLie1+AHqq/"
    "qBVu/Glq5sPL03TuQnosP0cdrHfwOkVOCjFVtj1Kf+/gTe9VJ6MdyhpJw0FAj2y9APW0wtWpyx9MzfwR6peUWwhj4vI8YlNKTPmD"
    "9ylalzn5K72ojDNlhtd4wAPaqUeRH5YmJ1utlt5ohgjf1KteY1JZLhQKFq8xy2u5jmdaajNwSgSKRhU91454WBLMTrHJmyKtBfAh"
    "W9DneUTC46zOTYsHYTn1c5xFaGJeMypfLMpuF+iBaYfcqHmBgUYcNUM13eAgjYv9qudGlC/pW+jYVR6qdmNek760YASmsaCH9qdc"
    "+ldjLXaJnS8Wi6WTjVimIO6sxi+7rH/7paivvQ6ILcpP4o8qOEzWdgC+bJk4sktUF4d/ehH/1Bl0a9hOPQDX4IOq/uZdsMl2I7XO"
    "3mZ4YJOspWkDidOcbmOhKEOqg7p4fo/VS7lI2+aOJSSqgeerRNf2OClq2K5axwM7x1JSDlsCFEM77FJ5WCf9zQXcXBhYYefK+eEJ"
    "lrI3xb5hLnADnUxNoU8rMbs381qkBXRycSYK2mekYJAK7ppzqNXGgmO5bvm3phPy0ugtfFG/qC+yK9O3WLK1Fz/fUj2Xf3D9usb6"
    "T9b7j45y91PS5C6ojunOl5UFD0G6KIMmmGZ5VcMLbFBKtCCj6phhaNfa0vB44UxHM+Gm2zIDHz0hlRAbEV+MHNvlw6p/UeMpgcvM"
    "8cUq9yN2s+3zqSDwgpMQoodsPWXxQSfZ7fxf4WMFJqtOmNmTucUJSwQRqsjlOM09BhUZVb5jh9Es2kBlpPrdZgOjiBky1xfLZkAt"
    "wPV1fDHbaqqAyvcTaq/KzO/eV1JigpR1MzQjtFthTPEDbtnVSNFKQ7GEUIdtPd1UoXfAa/QQbM9W8gU0EPQg2yXBUYZb1JnIph2K"
    "Yc+tchVGSafGODBgaFfCGXKFrhtAtbSsiS6mQJ18HNFJ5lEphIxqZQerEkAcn60MlWCaFYgURgOjUsqDSk/NOu7s+cpssSIC8jEx"
    "UEwqCeEZesW644pVsS2XK3mNoh2IrF1HjUggTB8GswtWvxzMN2mQnKanIO22JtJmWYaZ7qnKxEQYVBEJdJqYpcrZ1YhOpqRDjULf"
    "Tw6zWZJPUSmmViiNwOsy2uNAffFMGZpvIWJWRUUpYQQyG1HQ5ArdLo5fVo4PvDS1DM9d1XrTXQgnhyZBJbVGsPi6gIVs0tUzaGzT"
    "1z7MqH6NopPMjXiD0keB6uKO9HzuqqYOtMZRz1XPosagNKPaxK+BxKwiJJRKxntTFyAMGJppFJ+zpeyAFAiajmBLNoUIk4ZYTRNH"
    "jbc81JkzigHUcFAgxBm8L4AzwszAuiBsaVDhx2oKqQ3za4n+HK9qOlg5jQqDGZce7QixW0olb1ejI/LQm8XopIvZtvfDYTrUDuZo"
    "rxlUBRRqiIxwSyVC1mQ11Cgw4Zk+73hzqvK2ohHWNT1s1mr2IlBrEdNFzSj6R/48TUT45PKL78rPFp/zFU07rV9DmzRgh4Zlky7R"
    "OESQMuSKNtKKXHTbwCGLHI2SB2bE1TQIbbQ90Q15wqQcKQTvdMEwKNPO8mLomO3p79Ocdu2GOjzFaceiSu+XKfGBuqI2zs+6pzEr"
    "re0lT7r0DvBQvk2vfycG0R+6NEU93aeXQLwT0etf/Pmj4byNMgkd8/yJLT+gQammMDYrVVfYElyenXinWKosM7bEl5WTOaGp0Hab"
    "fGSDgF8YZ9X6MdwHM6N2CtRUlNRlTm7Rn/UzbJedRTldTm8sUI74ImrL8BbKN9GzTuUW/NVDE28oFvTWlCWQasy2xirLxpKLfwvL"
    "gqAnRUXtEqbH7+5qfYiLTYfqVfmzC8J7uBvEucH+nAkCDTWYyJw3qEeoRG3XbHClgtJAeYimMzTB2lE96z0/J0r2Twin/Uk3fXDW"
    "UpeoUZTyjgF7Qlqu5D7U7EgsyduWnjSsZiw3iGHYF0Q7e+RK04aOj7MOCob8Iz1wPjTIluG5ThubBEy2TsFSe8VyGve41ERgklEo"
    "GRoQMkYPMsn6mw/KS6nvY/B9TAOz5ZsG3p3LS7krWM5/GykvkRP5Cv06Ul5KHaCakBeAGITOYujQLyGSpuLisjC6qWkSxploHFCD"
    "mFrKKZcYLYX0Am+GVdvOhl7btXBDl8/LsGXIYFm5XGbJk0fJ5huGrylrh2czslk6BlWQJz6HhBg3O5Qt2vzLABpsE2Q4cixxFUom"
    "ph9gdexMnsRsNkt9Hvzcgcupkxx0SxAEHDQZozcYUrmBcJhiGDRfGYYiI5DDVuF/UEsDBBQAAAAIALpQSV0AxkdP2AcAABcdAAAa"
    "AAAAc2FtcGxlcy9kZXRhaWxfaW1hZ2VzLmpzb261mW1PG1cWx9/nU1zlFVnV+D4/9B1QaLME0s1GolJUofHMmLgFG2FXiVRVchKS"
    "UqBKKkhLWqCuSgOsLC1N3Q2rdbTfh7nzHfaM7THEOylh7GIYw3jmN+fec+75n3P5/BK6PFssVfzL76LLQb1h12r2fh3ZB/fs8jGy"
    "O43gn8d2v4rsz037aB0NBcfrwU7Tfn9ka9Xgp21kV7bs/jdXhlG4/iTY37O7cFwJ1vbO3rq8HbxYPjmqomCtDjfHrHDrafhgOzpv"
    "a/B2D76jm9b2hlG+UAFA9RSKwqePkX30tV3+dcjWngZrvyN7dHzyW81+f3hl+PI7MIpCxV8owyhuXULoc/iBM140KKm4NEZF18C5"
    "orPQHuqLAwRWhatNFD74Bv0l3Diw3zZQ+GwTbAhXjzuXgyHR1Xa1Hjxvds4VFpw5P34S/H27Ulksv5vNts4PL5TLrlccLvqV9oly"
    "dnHJmy0szGUpphIrrIjGhAmjNGeGKqO1dLjrYpn3fZ+Q4U8W59oPSknmRlADbCmY6pB9XztOmxyBP4bDF+/0ThNVFDg90wSODg6a"
    "9vkKePEY/ImCf1XtcQ3Z/7xE4e5KsFFDwVEj2DxE0RTtPrOre+jWycvjoLYC82jXtmGSa9Fv+9WP/7QppYZwqrHmmvJo3NzAlEqj"
    "81Ir4eZo+imNyRJrrbQgktCYbLij9ADIRlMiNcGYxGTHUaqPMIjJWjLBCNOGypjsUY+x/slCUCo1NVSrmKylJ8Q5AcYY5li/McBO"
    "jo+Cf8CifN4Ial9BHAUbEDxRqmm0ksjqnv3hABYm5IR6FIG3Tn5tBIeb4YOVnsAKfj4A3CACyzBMBKEUYooKLjVThEkn5zoQaUJr"
    "5qWeyi5ZGCI4gSdgHJMNlrn8AMhSUsnAV6pL1g7HuH8yGMuVEVIZE5OlIcb5Y/crLjTET4/7Zz76K5YEsvtWcH/L/rth/1uD1IGC"
    "V9WT42r7CBHxCrwfZWlbexJ+3URDkx+MTF69ksLt+dJ84e6w6+R9yofd0kK2NTqZv5O9c/cTMKTzNvvp7VmRNFEXuJ/3eb/s535z"
    "sZujw2yhmC/NEnyOFxmTGPd6MXxct80NFPxYDzfXkf2y2ZKIowP7avs1JxU/m58/Z13OzxfmSqV8vuD6Z0294+fAypmr0zfHb2Tb"
    "b3C4cT0xmt+GMTVydTo7ev3mzetTcHJi5sObWIx9kCwV6XnJyTY975wUq4jQTLNe76xXociKRBhWESTQkxevgh+/gjx6EOl2BoX3"
    "QInrrSW2EqVZWIU7DZRF18Y0VeSiDnyLVAJipwnkPQahCokVc00E6KqrsPClEdpNXDsXI2NKRaRYioqYjIXKJy6MC9osIPsZKrTA"
    "XZvznjcAMiRrzChTcIjJHnZx6lrglGxAYqIKhlLVJfuuTC0zp2QlGFRbmoHfYnJOE+4PwmaQWG0YBEhM9rFg/nkyA3WakT1LYHp0"
    "6r33+QwWFOJ65vrU+DSy3z2xy037cLtbeKxBowHL4odlqDygp7HfPURD741PX526csE1UK6UlvxhGBFU825p+NOlLCdKYSj3szgn"
    "lcY8n/GVdDPcd3nGcSjLaObiPESrJ5U3SxRnoNRRDMvEjPQHD4AKTeWUASIsqQzP5XjGSC4zruthwt2c8IyBBxjoQCQIsmQXfQB0"
    "GcYI7GW0ckmGOx7N5HyG4XngL2LcnPFceEBUEgrDIODOy1mcCSxwr8M+lNN/13gS3FSHng/Zp+tBbT2u/UavjYxNorHrExPj4xf0"
    "TaVUKpedVr6FHOyVo7grzOecYrZYWqrczjuun63crWRjA0Ynx5Im6G0wnl9xCvNnSbPJazktLNF1aWGJUpUWlpjA08ISK7G0sMSy"
    "Ki1MDRKm36Y/Jz3r5Ex/3hL3dt/dapPe1J2H9xrB7pZd23ljaw6Qk5d7UM0NrjuHBkowTTEXTJJO14gJyIbst4eOWjMQeU0jfeuS"
    "FZSpqTXoDFlDFmacSMy7ZMGw2z850k0WdXtcxmTQTZLrnwyzjCnTRJ/a7GrG0reUpx6EiSYGun+Nux4UxqRvg7tkmF9MIS6U6JIN"
    "U7rfvYqIrAmWSmKCu2RHcjUAMswGNhK+BDkzGzR9g31qMygzCDPljHZths/Oac2iO8DpvQliq47s7pcofLhud36PEgDU+tEm7sY2"
    "Cu/vhc+qyK48bm3GRaeRfbjZVtjJCwprNLLF0mKhmCstef5SK/+V2WIhl53/DF7+QqmYjXa5VPsoMctK43HuYMO49hMVIz3TcURe"
    "JwpHeqZPDB8wkzCcFzRRRtIzNc2p5ABPz/Q86XTS3psDEBouJf9Podq6tHuIOnvC3zai2Dv57SDa1oP4C5ePguVfoApvhOub4doR"
    "GgoOt4PdKnzc3Ui+aAXuu7CqOr116zcY6OIc1ZzmhJCMM2UwI9my75aKXtmZrxDKOv33xEh29KO/jV0bmRCYzCbWawOjJxZwg6JP"
    "vT+bGAQD5Ccu2oHNznn7ycIIjXv/r3NaEI3diGuinWZUE20uv763DAFnH60H+3sIgq6T9m51ro12SeLY+9O2lzUG5WCg1SzaXYbO"
    "nMHLUdSXOs/zSqRX7FOyEFowqDU4j8lC+iR9zRWTOSPRbjA06NGOQkQ2GJpNk179ujaDvZgwDd2qjm3WjPSxJd4lSy04VBkEqq+Y"
    "7Cih+OtxdgnevvgfUEsBAhQDFAAAAAgAwUhJXYOlD78nEwAAKDcAAAsAAAAAAAAAAAAAAKSBAAAAAG1hdGNoaW5nLnB5UEsBAhQD"
    "FAAAAAgAqSFJXRLFrOH5AgAArgYAABQAAAAAAAAAAAAAAKSBUBMAAGRhdGEvdGFnX3dlaWdodC5qc29uUEsBAhQDFAAAAAgA5yJJ"
    "XZstpDVxAwAAagoAABIAAAAAAAAAAAAAAKSBexYAAGRhdGEva2V5d29yZHMuanNvblBLAQIUAxQAAAAIAK0hSV387wN5IQMAAOkG"
    "AAARAAAAAAAAAAAAAACkgRwaAABkYXRhL3JlYXNvbnMuanNvblBLAQIUAxQAAAAIAK4hSV2onCjKVAEAANkBAAAQAAAAAAAAAAAA"
    "AACkgWwdAABkYXRhL2NvbmZpZy5qc29uUEsBAhQDFAAAAAgA5iJJXeMzRz9gAgAAAgUAABEAAAAAAAAAAAAAAKSB7h4AAGRhdGEv"
    "YmFsYW5jZS5qc29uUEsBAhQDFAAAAAgAAmhJXRbtySbaCQAAwRQAABcAAAAAAAAAAAAAAKSBfSEAAHRvb2xzL29jcl9leHBlcmlt"
    "ZW50LnB5UEsBAhQDFAAAAAgAulBJXQDGR0/YBwAAFx0AABoAAAAAAAAAAAAAAKSBjCsAAHNhbXBsZXMvZGV0YWlsX2ltYWdlcy5q"
    "c29uUEsFBgAAAAAIAAgABAIAAJwzAAAAAA=="
)
os.makedirs('/content/algorithm', exist_ok=True)
zipfile.ZipFile(io.BytesIO(base64.b64decode(BUNDLE))).extractall('/content/algorithm')
print(sorted(os.listdir('/content/algorithm')))

In [ ]:
# 3. 3개 상품만 먼저 (첫 실행은 모델을 받느라 몇 분 걸림)
%cd /content/algorithm
!python tools/ocr_experiment.py --limit 3 --keep

**여기서 확인할 것**
- 상품마다 `글자수=`가 0이 아닌지
- `이름만=[...]` 대비 `이름+OCR=[...]`에서 태그가 늘었는지
- `[실패]`가 많이 나오면 이미지 다운로드가 막힌 것입니다. 그 줄을 알려 주세요.
- 모델 다운로드 오류가 나면 아래 칸(환경변수)을 먼저 실행하고 3번 칸을 다시 실행하세요.

In [ ]:
# (필요할 때만) 모델 다운로드 서버를 Baidu 쪽으로 바꾸기
import os
os.environ['PADDLE_PDX_MODEL_SOURCE'] = 'BOS'

In [ ]:
# 4. 전체 12개 상품 실행
!python tools/ocr_experiment.py

In [ ]:
# 읽힌 글을 눈으로 확인 (상품별 앞 400자)
import json
for r in json.load(open('samples/ocr_result.json', encoding='utf-8')):
    print('=====', r['id'], r['name'], '| 핏:', r['fit'], '| 글자수:', r['ocr_chars'])
    print('이름만:', r['tags_name_only'], '-> 이름+OCR:', r['tags_with_ocr'])
    print(r['ocr_text'][:400])
    print()

In [ ]:
# 5. 결과 파일 내려받기 -> samples 폴더에 넣어서 보내 주세요
from google.colab import files
files.download('samples/ocr_result.json')